# SimFL: Horizontal Federated GBDT (Li, Wen & He, AAAI 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima0003/EVCS/blob/claude/federated-gbdt-implementation-qb8cuv/SimFL.ipynb)

Re-implementation of **"Practical Federated Gradient Boosting Decision Trees"** ([arXiv:1911.04206](https://arxiv.org/abs/1911.04206)) for intrusion detection.

* **Preprocessing (LSH):** every party hashes its samples with `L` shared p-stable hash functions `h(x) = floor((a·x + b)/r)`.
  Parties share only hash values and sample IDs. Each sample's *similar instance* at another party is the sample there
  with the most identical hash values.
* **Training (Weighted Gradient Boosting):** parties take turns building trees. Every other party sends the builder the
  sums of its gradients and Hessians, grouped by similar instance. The builder grows a tree on its own samples using these
  weighted gradients and broadcasts it.

Supports binary (benign vs attack) and multi-class (softmax), IID and Dirichlet non-IID partitions, the SOLO and ALL-IN
baselines, and efficiency metrics (time, communication, estimated network time, inference latency, model size).
Every run is saved as JSON plus one CSV row.

**How to run in Colab:** upload `CICIDS2017_sample_km.csv`, `IoT_Network_Intrusion_Dataset_0.1.csv` and
`CICEVSE2024_12classes_sample.csv` to `/content` (the session's working directory), then *Runtime → Run all*.

### Implementation notes: where the paper is ambiguous or conflicts with the instructions

The paper's text could not be re-read while building this notebook (network restrictions), so details were checked against the
authors' official code, [Xtra-Computing/SimFL](https://github.com/Xtra-Computing/SimFL).

| Topic | Paper / official code | This notebook |
|---|---|---|
| Hash vector `a` | Generic p-stable; the official code uses **Cauchy** | **Gaussian** (as instructed); `cauchy` is available |
| Normalization before hashing | None (raw features hashed) | Z-score from a *federated* scaler: parties exchange per-feature `(n, mean, M2)`. This equals the scaler fitted on the pooled training data and costs 1 extra preprocessing round, counted separately. Trees use raw features. |
| `L`, `r` | Official code: 40 and 4.0. As I recall, the paper says `L = min(40, d−1)`. | `L = min(40, d−1)`, `r = 4.0` (configurable) |
| Ties among equally similar samples | Paper: no rule found. Official code: lowest ID. | Uniform random (seeded); `lowest_id` is available |
| Sample with no hash collision | Official code silently maps it to sample 0 | Its gradient is not sent; the count is logged. This is rare. |
| Hash buckets | Official code uses `hash % 500` | Raw hash values (no false collisions) |
| Tree schedule | Official entrypoint: each party builds a contiguous block of trees ∝ its size. The exact-split path uses round robin. | `proportional_blocks` (default) or `round_robin` |
| What is aggregated | WGB needs gradients **and** Hessians, and the official code sums both | Both |
| Loss | Paper: generic. Official code: squared error on 0/1 labels. | Logistic (binary) / softmax (multi-class), identical to XGBoost's built-ins |
| Hyperparameters | Official code: 50 trees, depth 8, η=0.1, λ=1, γ=1. Its histogram path never applies η. | 50 rounds, depth 8, η=0.1, λ=1, γ=1, 256 bins for every method |
| Split candidates | Equal-width bins over the *global* min/max (leaks ranges) | XGBoost quantile sketch of the builder's own data |
| Tree learner | — | XGBoost, one round per tree with a custom objective returning the precomputed weighted `(G, H)`. This is exactly WGB: one party reproduces centralized XGBoost (see the sanity checks). `base_margin` is unnecessary because the objective ignores `preds`. |
| Multi-class | Not in the paper | K-vectors of `(g, h)` per sample are aggregated the same way. One round = K trees. |

Phase timings add up the work of all simulated parties, because parties run sequentially in one process. Network cost is
estimated as `rounds × latency + bytes / bandwidth`.

# Setup

In [ ]:
# Colab ships xgboost; make sure it is >= 2.0 (needed for multi-class custom objectives)
!pip -q install "xgboost>=2.0"

In [ ]:
import os
N_THREADS = os.cpu_count() or 2
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = str(N_THREADS)

import copy, json, platform, sys, time, warnings
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
import xgboost as xgb
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
ENV = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgb.__version__, "n_threads": N_THREADS}
print(ENV)

# Configuration

In [ ]:
RESULTS_DIR = Path("results/simfl")

DEFAULTS = {
    "n_trees": 50, "max_depth": 8, "learning_rate": 0.1, "reg_lambda": 1.0, "gamma": 1.0,
    "min_child_weight": 1.0, "max_bin": 256,
    "lsh_L": None, "lsh_max_L": 40, "lsh_r": 4.0, "lsh_p_stable": "gaussian", "lsh_tie_break": "random",
    "schedule": "proportional_blocks",
    "test_size": 0.20, "split_seed": 42, "min_party_size": 20,
    "latency_ms": 20.0, "bandwidth_mbps": 100.0,
    "n_single": 1000, "n_warmup": 100,
    "n_threads": N_THREADS,
}

# Full experiment grid (Section "Full grid" of each dataset)
RUN_FULL_GRID = True
GRID = {
    "modes": ["multiclass"],                    # add "binary" if time allows (needs a benign label)
    "partitions": ["iid", "dirichlet"],
    "dirichlet_alphas": [0.5],
    "n_parties": [10],
    "seeds": [0, 1, 2],
    "methods": ["SimFL", "SOLO", "ALL-IN"],
    "resume": True,                             # skip runs whose JSON already exists
}

# Utility funcs

In [ ]:
class Timer:
    """Accumulates wall-clock seconds per named phase."""
    def __init__(self):
        self.t = defaultdict(float)

    @contextmanager
    def __call__(self, key):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self.t[key] += time.perf_counter() - t0


def payload_nbytes(obj) -> int:
    if isinstance(obj, np.ndarray):
        return int(obj.nbytes)
    if isinstance(obj, (bytes, bytearray)):
        return len(obj)
    if isinstance(obj, (int, float, np.integer, np.floating)):
        return 8
    if isinstance(obj, dict):
        return sum(payload_nbytes(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(payload_nbytes(v) for v in obj)
    raise TypeError(f"unsupported payload type {type(obj)}")


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o

# Communicator

Every exchange between parties goes through it. `send` deep-copies the payload into the receiver's mailbox and counts its
exact size. A **round** is one synchronous exchange (parties → peer(s) → back). Preprocessing rounds (scaler statistics,
hash exchange) are counted separately from training rounds.

In [ ]:
class Communicator:
    def __init__(self, n_parties: int):
        self.n_parties = n_parties
        self._mailbox = {p: [] for p in range(n_parties)}
        self.rounds = defaultdict(int)     # (phase, step) -> rounds
        self.bytes = defaultdict(int)      # (phase, tag)  -> bytes
        self.messages = 0
        self._phase = None

    def new_round(self, phase: str, step: str):
        self._phase = phase
        self.rounds[(phase, step)] += 1

    def send(self, src: int, dst: int, tag: str, payload):
        assert src != dst and self._phase is not None
        self.bytes[(self._phase, tag)] += payload_nbytes(payload)
        self.messages += 1
        self._mailbox[dst].append((src, tag, copy.deepcopy(payload)))

    def broadcast(self, src: int, tag: str, payload):
        for dst in range(self.n_parties):
            if dst != src:
                self.send(src, dst, tag, payload)

    def receive(self, dst: int, tag: str):
        mine = [(s, p) for s, t, p in self._mailbox[dst] if t == tag]
        self._mailbox[dst] = [m for m in self._mailbox[dst] if m[1] != tag]
        return sorted(mine, key=lambda m: m[0])

    def summary(self) -> Dict[str, int]:
        r = lambda ph, st=None: sum(v for (p, s), v in self.rounds.items() if p == ph and (st is None or s == st))
        b = lambda ph, tg=None: sum(v for (p, t), v in self.bytes.items() if p == ph and (tg is None or t == tg))
        out = {"scaler_rounds": r("preprocessing", "scaler_stats"),
               "hash_exchange_rounds": r("preprocessing", "lsh_hashes"),
               "preprocessing_rounds": r("preprocessing"),
               "training_rounds": r("training"),
               "scaler_bytes": b("preprocessing", "scaler_stats"),
               "hash_exchange_bytes": b("preprocessing", "lsh_hashes"),
               "preprocessing_bytes": b("preprocessing"),
               "gradient_bytes": b("training", "aggregated_gradients"),
               "tree_bytes": b("training", "tree"),
               "training_bytes": b("training"),
               "messages": self.messages}
        out["total_rounds"] = out["preprocessing_rounds"] + out["training_rounds"]
        out["total_bytes"] = out["preprocessing_bytes"] + out["training_bytes"]
        return out


ZERO_COMM = Communicator(1).summary()

# LSH preprocessing (paper Sec. "Preprocessing stage")

`most_similar` returns, for each query sample, the target sample with the most identical hash values. Rows with identical
signatures are de-duplicated, and match counts come from a chunked sparse product of one-hot `(table, hash value)` indicators.
Random ties are uniform over the tied *samples*.

In [ ]:
class PStableLSH:
    def __init__(self, n_features: int, L: int, r: float, seed: int, p_stable: str = "gaussian"):
        rng = np.random.default_rng(seed)                    # shared seed -> identical functions at every party
        if p_stable == "gaussian":
            self.a = rng.standard_normal((n_features, L))
        elif p_stable == "cauchy":
            self.a = rng.standard_cauchy((n_features, L))
        else:
            raise ValueError(p_stable)
        self.b = rng.uniform(0.0, r, L)
        self.r = r

    def hash(self, Z: np.ndarray) -> np.ndarray:
        h = np.floor((Z @ self.a + self.b) / self.r)
        assert np.abs(h).max() < 2**31
        return h.astype(np.int32)


def most_similar(H_q, H_t, rng, tie_break="random", chunk=2048):
    """For every row of H_q: index of the row of H_t sharing the most hash values (-1 if none), and that count."""
    L = H_q.shape[1]
    Uq, inv_q = np.unique(H_q, axis=0, return_inverse=True)
    Ut, inv_t, cnt_t = np.unique(H_t, axis=0, return_inverse=True, return_counts=True)
    inv_q, inv_t = inv_q.ravel(), inv_t.ravel()
    cq, ct, off = [], [], 0
    for l in range(L):                                       # shared (table, hash value) column space
        vals, codes = np.unique(np.concatenate([Uq[:, l], Ut[:, l]]), return_inverse=True)
        cq.append(codes[:len(Uq)] + off); ct.append(codes[len(Uq):] + off); off += len(vals)
    cq, ct = np.stack(cq, 1), np.stack(ct, 1)
    onehot = lambda c, n: sp.csr_matrix((np.ones(c.size, np.int32), c.ravel(), np.arange(0, c.size + 1, L)),
                                        shape=(n, off))
    Aq, AtT = onehot(cq, len(Uq)), onehot(ct, len(Ut)).T.tocsr()

    order_t = np.argsort(inv_t, kind="stable")
    start_t = np.concatenate([[0], np.cumsum(cnt_t)])
    order_q = np.argsort(inv_q, kind="stable")
    start_q = np.concatenate([[0], np.cumsum(np.bincount(inv_q, minlength=len(Uq)))])
    min_id_t = np.array([order_t[start_t[s]:start_t[s + 1]].min() for s in range(len(Ut))])

    out = np.full(len(H_q), -1, np.int64)
    n_match = np.zeros(len(H_q), np.int32)
    for s in range(0, len(Uq), chunk):
        e = min(s + chunk, len(Uq))
        C = (Aq[s:e] @ AtT).toarray()                        # matching-hash counts
        mx = C.max(1)
        inst = order_q[start_q[s]:start_q[e]]                # query samples whose signature is in this chunk
        rows = inv_q[inst] - s
        tied = (C == mx[:, None]) & (mx[:, None] > 0)
        if tie_break == "lowest_id":
            sig = np.where(tied, min_id_t[None, :], np.iinfo(np.int64).max).argmin(1)[rows]
            member_id = min_id_t[sig]
        else:
            cw = np.cumsum(np.where(tied, cnt_t[None, :], 0), 1, dtype=np.float64)
            u = rng.random(len(inst)) * cw[rows, -1]
            sig = (cw[rows] > u[:, None]).argmax(1)
            k = np.minimum((rng.random(len(inst)) * cnt_t[sig]).astype(np.int64), cnt_t[sig] - 1)
            member_id = order_t[start_t[sig] + k]
        ok = mx[rows] > 0
        out[inst[ok]] = member_id[ok]
        n_match[inst] = mx[rows]
    return out, n_match


def combine_moments(stats):
    """Chan et al. parallel combination of per-party (n, mean, M2) -> global mean, std."""
    n, mean, m2 = 0, None, None
    for s in stats:
        nb, mb, m2b = s["n"], s["mean"].astype(np.float64), s["m2"].astype(np.float64)
        if n == 0:
            n, mean, m2 = nb, mb.copy(), m2b.copy()
            continue
        delta, tot = mb - mean, n + nb
        mean = mean + delta * nb / tot
        m2 = m2 + m2b + delta**2 * n * nb / tot
        n = tot
    std = np.sqrt(m2 / n)
    return mean, np.where(std > 0, std, 1.0)

# Losses and XGBoost helpers

The gradients and Hessians match XGBoost's `binary:logistic` / `multi:softprob`. Each tree is one `xgb.train` round
with a custom objective that returns the precomputed weighted `(G, H)`. The global model is an ordered list of one-round
boosters, merged at the end into a single booster (with a fallback to summing the list if the XGBoost version's JSON layout differs).

In [ ]:
def grad_hess(margin, y, mode):
    if mode == "binary":
        p = sigmoid(margin)
        return p - y, np.maximum(p * (1.0 - p), 1e-16)
    p = softmax(margin)
    g = p.copy()
    g[np.arange(len(y)), y] -= 1.0
    return g, np.maximum(2.0 * p * (1.0 - p), 1e-16)


def xgb_params(cfg, mode, n_classes, seed, custom):
    p = {"tree_method": "hist", "max_depth": cfg["max_depth"], "eta": cfg["learning_rate"],
         "reg_lambda": cfg["reg_lambda"], "gamma": cfg["gamma"], "min_child_weight": cfg["min_child_weight"],
         "max_bin": cfg["max_bin"], "nthread": cfg["n_threads"], "seed": seed, "verbosity": 0}
    if custom:
        p.update(base_score=0.0, disable_default_eval_metric=1)
        if mode == "multiclass":
            p["num_class"] = n_classes
    elif mode == "binary":
        p.update(objective="binary:logistic", base_score=0.5)        # margin starts at 0, like SimFL
    else:
        p.update(objective="multi:softprob", num_class=n_classes, base_score=0.0)
    return p


def load_booster(raw, n_threads):
    b = xgb.Booster()
    b.load_model(bytearray(raw))
    b.set_param({"nthread": n_threads})
    return b


class SummedEnsemble:
    """Fallback global model: an ordered list of boosters whose margins are summed."""
    def __init__(self, boosters):
        self.boosters = boosters

    def inplace_predict(self, X, predict_type="margin"):
        return sum(b.inplace_predict(X, predict_type="margin") for b in self.boosters)


def merge_boosters(boosters, n_threads):
    base = json.loads(boosters[0].save_raw("json"))
    model = base["learner"]["gradient_booster"]["model"]
    trees, info, indptr = [], [], [0]
    for b in boosters:
        m = json.loads(b.save_raw("json"))["learner"]["gradient_booster"]["model"]
        for tree, cls in zip(m["trees"], m["tree_info"]):
            trees.append(dict(tree, id=len(trees))); info.append(cls)
        indptr.append(len(trees))
    model.update(trees=trees, tree_info=info, iteration_indptr=indptr)
    model["gbtree_model_param"]["num_trees"] = str(len(trees))
    return load_booster(json.dumps(base).encode(), n_threads)


def build_global_model(boosters, X_check, n_threads):
    """Merge into one booster; verify against the summed list, else fall back to SummedEnsemble."""
    ens = SummedEnsemble(boosters)
    try:
        merged = merge_boosters(boosters, n_threads)
        Xc = np.ascontiguousarray(X_check[:256], dtype=np.float32)
        if np.allclose(merged.inplace_predict(Xc, predict_type="margin"), ens.inplace_predict(Xc), atol=1e-4):
            return merged
    except Exception as e:
        print("  merge failed, using SummedEnsemble:", e)
    return ens


def model_stats(model):
    boosters = model.boosters if isinstance(model, SummedEnsemble) else [model]
    trees = [t for b in boosters for t in json.loads(b.save_raw("json"))["learner"]["gradient_booster"]["model"]["trees"]]
    return {"bytes": int(sum(len(b.save_raw("ubj")) for b in boosters)), "n_trees": len(trees),
            "total_nodes": int(sum(int(t["tree_param"]["num_nodes"]) for t in trees))}

# Party

Holds its data privately and exposes only the protocol steps. Everything another party learns arrives through the `Communicator`.

In [ ]:
class Party:
    def __init__(self, pid, X, y, mode, n_classes, comm, timer, cfg, seed):
        self.pid, self.mode, self.n_classes = pid, mode, n_classes
        self._X = np.ascontiguousarray(X, dtype=np.float32)          # private
        self._y = y                                                  # private
        self._comm, self._timer, self._cfg = comm, timer, cfg
        self._rng = np.random.default_rng([seed, 1000 + pid])
        self._dmat = xgb.DMatrix(self._X, nthread=cfg["n_threads"])
        self._margin = np.zeros((len(y),) if mode == "binary" else (len(y), n_classes))
        self.models = []          # this party's copy of the global model
        self.similar = {}         # other pid -> similar-instance index at that party, per own sample
        self.n_matching = {}
        self.peer_sizes = {}

    @property
    def n_samples(self):
        return len(self._y)

    # ---------- preprocessing ----------
    def share_scaler_stats(self):
        X = self._X.astype(np.float64)
        mean = X.mean(0)
        self._own_stats = {"n": len(X), "mean": mean, "m2": ((X - mean) ** 2).sum(0)}
        self._comm.broadcast(self.pid, "scaler_stats", self._own_stats)

    def fit_global_scaler(self):
        stats = dict(self._comm.receive(self.pid, "scaler_stats"))
        stats[self.pid] = self._own_stats
        self._mu, self._sd = combine_moments([stats[p] for p in sorted(stats)])

    def share_hashes(self, lsh):
        self._H = lsh.hash((self._X - self._mu) / self._sd)
        self._comm.broadcast(self.pid, "lsh_hashes",
                             {"ids": np.arange(self.n_samples, dtype=np.int32), "hashes": self._H})

    def find_similar(self, tie_break):
        for src, msg in self._comm.receive(self.pid, "lsh_hashes"):
            idx, n_match = most_similar(self._H, msg["hashes"], self._rng, tie_break)
            self.similar[src] = np.where(idx >= 0, msg["ids"][np.maximum(idx, 0)], -1)
            self.n_matching[src] = n_match
            self.peer_sizes[src] = len(msg["ids"])

    # ---------- training ----------
    def local_grad_hess(self):
        with self._timer("local_gradients"):
            self._g, self._h = grad_hess(self._margin, self._y, self.mode)

    def send_aggregated(self, builder):
        with self._timer("aggregation_comm"):
            idx = self.similar[builder]
            ok = idx >= 0
            tgt, inv = np.unique(idx[ok], return_inverse=True)
            g, h = self._g[ok].reshape(ok.sum(), -1), self._h[ok].reshape(ok.sum(), -1)
            G = np.stack([np.bincount(inv, g[:, k], len(tgt)) for k in range(g.shape[1])], 1)
            Hs = np.stack([np.bincount(inv, h[:, k], len(tgt)) for k in range(h.shape[1])], 1)
            K, n_b = G.shape[1], self.peer_sizes[builder]
            if len(tgt) * (4 + 8 * K) < n_b * 8 * K:                      # sparse encoding is smaller
                payload = {"idx": tgt.astype(np.int32), "g": G.astype(np.float32), "h": Hs.astype(np.float32)}
            else:
                Gd, Hd = np.zeros((n_b, K), np.float32), np.zeros((n_b, K), np.float32)
                Gd[tgt], Hd[tgt] = G, Hs
                payload = {"g": Gd, "h": Hd}
            self._comm.send(self.pid, builder, "aggregated_gradients", payload)

    def build_tree(self, params):
        with self._timer("aggregation_comm"):
            G = self._g.reshape(self.n_samples, -1).astype(np.float64)
            H = self._h.reshape(self.n_samples, -1).astype(np.float64)
            for _, msg in self._comm.receive(self.pid, "aggregated_gradients"):
                if "idx" in msg:
                    G[msg["idx"]] += msg["g"]; H[msg["idx"]] += msg["h"]
                else:
                    G += msg["g"]; H += msg["h"]
            if self.mode == "binary":
                G, H = G[:, 0], H[:, 0]
        with self._timer("tree_building"):
            booster = xgb.train(params, self._dmat, num_boost_round=1, obj=lambda preds, dtrain: (G, H))
        with self._timer("aggregation_comm"):
            self._comm.broadcast(self.pid, "tree", {"model": booster.save_raw("ubj")})
        self._apply(booster)

    def receive_tree(self):
        with self._timer("aggregation_comm"):
            (_, msg), = self._comm.receive(self.pid, "tree")
            booster = load_booster(msg["model"], self._cfg["n_threads"])
        self._apply(booster)

    def _apply(self, booster):
        with self._timer("prediction_update"):
            self.models.append(booster)
            self._margin += booster.inplace_predict(self._X, predict_type="margin")

# SimFL training (paper Algorithm 2, Weighted Gradient Boosting)

In [ ]:
def tree_schedule(sizes, n_rounds, kind):
    """Builder party for every boosting round."""
    M = len(sizes)
    if kind == "round_robin":
        return [t % M for t in range(n_rounds)]
    starts = [0]                    # official SimFL: party i builds floor(T * n_i / N) consecutive trees
    for i in range(1, M):
        starts.append(starts[-1] + n_rounds * sizes[i - 1] // sum(sizes))
    return [max(i for i in range(M) if starts[i] <= t) for t in range(n_rounds)]


def train_simfl(parts, mode, n_classes, cfg, seed, log_every=0):
    """parts: list of (X, y) per party. Returns (global model, timings, communication, extra info)."""
    t_start, M, timer = time.perf_counter(), len(parts), Timer()
    comm = Communicator(M)
    with timer("setup"):
        parties = [Party(i, X, y, mode, n_classes, comm, timer, cfg, seed) for i, (X, y) in enumerate(parts)]
    d = parts[0][0].shape[1]
    L = cfg["lsh_L"] or min(cfg["lsh_max_L"], d - 1)

    with timer("lsh_preprocessing"):
        comm.new_round("preprocessing", "scaler_stats")
        for p in parties: p.share_scaler_stats()
        for p in parties: p.fit_global_scaler()
        lsh = PStableLSH(d, L, cfg["lsh_r"], seed, cfg["lsh_p_stable"])
        comm.new_round("preprocessing", "lsh_hashes")
        for p in parties: p.share_hashes(lsh)
        for p in parties: p.find_similar(cfg["lsh_tie_break"])

    params = xgb_params(cfg, mode, n_classes, seed, custom=True)
    schedule = tree_schedule([p.n_samples for p in parties], cfg["n_trees"], cfg["schedule"])
    for t, builder in enumerate(schedule):
        comm.new_round("training", "boosting_round")
        for p in parties: p.local_grad_hess()
        for p in parties:
            if p.pid != builder: p.send_aggregated(builder)
        parties[builder].build_tree(params)
        for p in parties:
            if p.pid != builder: p.receive_tree()
        if log_every and (t + 1) % log_every == 0:
            print(f"  Boosting round {t + 1}/{len(schedule)} done (built by party {builder})")

    with timer("model_merge"):
        model = build_global_model(parties[0].models, parts[0][0], cfg["n_threads"])
    timer.t["train_total"] = time.perf_counter() - t_start

    matches = [m for p in parties for m in p.n_matching.values()]
    extra = {"L": L, "r": cfg["lsh_r"], "p_stable": cfg["lsh_p_stable"], "tie_break": cfg["lsh_tie_break"],
             "schedule": cfg["schedule"], "trees_per_party": np.bincount(schedule, minlength=M).tolist(),
             "unmatched_samples": int(sum((p.similar[q] < 0).sum() for p in parties for q in p.similar)),
             "mean_matching_hashes": float(np.concatenate(matches).mean()) if matches else None}
    return model, dict(timer.t), comm.summary(), extra

# Baselines

* **ALL-IN**: centralized XGBoost on the pooled training data.
* **SOLO**: each party trains the same XGBoost on its own data. Metrics are averaged over parties.

Both use the same hyperparameters as SimFL.

In [ ]:
def train_xgb_local(X, y, mode, n_classes, cfg, seed):
    t0, timer = time.perf_counter(), Timer()
    with timer("setup"):
        dm = xgb.DMatrix(np.ascontiguousarray(X, dtype=np.float32), label=y, nthread=cfg["n_threads"])
    with timer("tree_building"):
        model = xgb.train(xgb_params(cfg, mode, n_classes, seed, custom=False), dm, cfg["n_trees"])
    timer.t["train_total"] = time.perf_counter() - t0
    return model, dict(timer.t)

# Partitioning

In [ ]:
def partition_iid(y, n_clients, seed):
    if n_clients == 1:
        return [np.arange(len(y))]
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=seed)
    return [np.sort(te) for _, te in skf.split(np.zeros(len(y)), y)]


def partition_dirichlet(y, n_clients, alpha, seed, min_size=20, max_tries=10_000):
    """Per-class Dirichlet(alpha) proportions; re-drawn until every client has >= min_size samples."""
    rng = np.random.default_rng(seed)
    for _ in range(max_tries):
        parts = [[] for _ in range(n_clients)]
        for c in np.unique(y):
            idx = rng.permutation(np.flatnonzero(y == c))
            cuts = (np.cumsum(rng.dirichlet(np.full(n_clients, alpha))) * len(idx)).astype(int)[:-1]
            for i, chunk in enumerate(np.split(idx, cuts)):
                parts[i].extend(chunk.tolist())
        if min(len(p) for p in parts) >= min_size:
            return [np.sort(np.array(p, dtype=np.int64)) for p in parts]
    raise RuntimeError(f"no Dirichlet(alpha={alpha}) draw gave every client >= {min_size} samples")


def make_partition(y, partition_mode, n_clients, seed, alpha, cfg):
    if partition_mode == "iid":
        return partition_iid(y, n_clients, seed)
    return partition_dirichlet(y, n_clients, alpha, seed, cfg["min_party_size"])

# Data preparation

* Non-numeric columns other than the label are dropped (with a message), as are the configured `drop_cols`.
* Non-finite values are replaced by 0.
* Classes are re-indexed. When a `benign_label` is given it becomes class 0, which enables binary mode and the FPR.
* The test set is held out, centralized and stratified (`split_seed`), and identical for every run.

In [ ]:
def prepare_dataset(df, name, label_col, benign_label=None, drop_cols=(), class_names=None, cfg=DEFAULTS):
    df = df.drop(columns=[c for c in drop_cols if c in df.columns and c != label_col])
    y_raw = df[label_col]
    X = df.drop(columns=[label_col])
    non_num = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    if non_num:
        print(f"[{name}] dropping non-numeric columns: {non_num}")
        X = X.drop(columns=non_num)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    values = sorted(y_raw.unique(), key=lambda v: str(v) if not isinstance(v, (int, np.integer)) else f"{v:012d}")
    if benign_label is not None:
        assert benign_label in set(values), f"benign_label {benign_label!r} not found in {label_col}"
        values = [benign_label] + [v for v in values if v != benign_label]
    y = y_raw.map({v: i for i, v in enumerate(values)}).to_numpy(np.int64)
    names = [str(class_names.get(v, v)) if class_names else str(v) for v in values]

    idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=cfg["test_size"],
                                      random_state=cfg["split_seed"], stratify=y)
    Xn = X.to_numpy(np.float32)
    return {"name": name, "label_col": label_col, "has_benign": benign_label is not None,
            "class_names": names, "feature_names": list(X.columns), "n_features": Xn.shape[1],
            "drop_cols": list(drop_cols),
            "X_train": Xn[idx_tr], "y_train": y[idx_tr], "X_test": Xn[idx_te], "y_test": y[idx_te]}


def labels_for_mode(y, mode):
    return (y != 0).astype(np.int64) if mode == "binary" else y


def class_names_for_mode(ds, mode):
    return ["benign", "attack"] if mode == "binary" else ds["class_names"]

# Evaluation metrics

* **Inference.** All models use the same `predict_labels` (float32 conversion → margin → argmax/threshold).
  *Batch* = full test set divided by n. *Single* = median of `n_single` one-row predictions after `n_warmup` warm-up calls.
* **Model size.** Serialized booster (UBJSON) bytes, number of trees, total nodes.
* **Network estimate.** `rounds × latency + bytes × 8 / bandwidth`.
* **FPR.** Fraction of truly benign test samples predicted as any attack class. Only computed when a benign class is known.

In [ ]:
def predict_labels(model, X, mode):
    margin = model.inplace_predict(np.ascontiguousarray(X, dtype=np.float32), predict_type="margin")
    return (margin > 0).astype(np.int64) if mode == "binary" else margin.argmax(1)


def inference_timing(model, X_test, mode, cfg):
    t0 = time.perf_counter()
    predict_labels(model, X_test, mode)
    batch = (time.perf_counter() - t0) / len(X_test)
    rows = np.random.default_rng(0).integers(0, len(X_test), cfg["n_warmup"] + cfg["n_single"])
    lat = []
    for k, i in enumerate(rows):
        t0 = time.perf_counter()
        predict_labels(model, X_test[i:i + 1], mode)
        if k >= cfg["n_warmup"]:
            lat.append(time.perf_counter() - t0)
    return {"batch_per_sample_s": batch, "single_median_s": float(np.median(lat)), "n_single": len(lat)}


def detection_metrics(y_true, y_pred, names, has_benign):
    labels = list(range(len(names)))
    prec, rec, _, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    benign = y_true == 0
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
            "macro_precision": float(prec.mean()), "macro_recall": float(rec.mean()),
            "fpr": float((y_pred[benign] != 0).mean()) if has_benign else None,
            "precision": dict(zip(names, prec.tolist())), "recall": dict(zip(names, rec.tolist())),
            "support": dict(zip(names, sup.tolist())),
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()}


def network_estimate(comm, cfg):
    lat, bw = cfg["latency_ms"] / 1e3, cfg["bandwidth_mbps"] * 1e6
    t = lambda rounds, nbytes: rounds * lat + nbytes * 8 / bw
    pre, tr = t(comm["preprocessing_rounds"], comm["preprocessing_bytes"]), t(comm["training_rounds"], comm["training_bytes"])
    return {"latency_ms": cfg["latency_ms"], "bandwidth_mbps": cfg["bandwidth_mbps"],
            "preprocessing_s": pre, "training_s": tr, "total_s": pre + tr}


def evaluate(model, mode, ds, cfg):
    y_test = labels_for_mode(ds["y_test"], mode)
    y_pred = predict_labels(model, ds["X_test"], mode)
    q = detection_metrics(y_test, y_pred, class_names_for_mode(ds, mode), ds["has_benign"])
    return q, inference_timing(model, ds["X_test"], mode, cfg), model_stats(model), y_pred


def mean_records(recs):
    """Element-wise mean of nested numeric dicts (SOLO averages over parties)."""
    out = {}
    for k, v in recs[0].items():
        if isinstance(v, dict):
            out[k] = mean_records([r[k] for r in recs])
        elif v is None or isinstance(v, bool):
            out[k] = v
        elif isinstance(v, (int, float, np.integer, np.floating)):
            out[k] = float(np.mean([r[k] for r in recs]))
        elif k == "confusion_matrix":
            out[k] = np.mean([r[k] for r in recs], 0).tolist()
        else:
            out[k] = v
    return out

# Run records (JSON + CSV)

Every run is saved to `results/simfl/<dataset>/runs/<run_id>.json`. `runs.csv` holds one flattened row per run.

In [ ]:
TIMING_KEYS = ["train_total", "setup", "lsh_preprocessing", "local_gradients", "tree_building",
               "aggregation_comm", "prediction_update", "model_merge"]


def run_id_for(ds, method, mode, partition_mode, alpha, n_clients, seed):
    part = "pooled" if method == "ALL-IN" else ("iid" if partition_mode == "iid" else f"dirichlet_a{alpha}")
    return f"{ds['name']}__{mode}__{method}__{part}__M{n_clients}__seed{seed}"


def run_method(method, ds, mode, partition_mode, alpha, party_idx, seed, cfg, log_every=0):
    """Train + evaluate one method on one partition. Returns (record, model, y_pred)."""
    if mode == "binary" and not ds["has_benign"]:
        raise ValueError(f"{ds['name']}: binary mode needs benign_label")
    K = len(class_names_for_mode(ds, mode))
    y_tr = labels_for_mode(ds["y_train"], mode)
    n_clients = 1 if method == "ALL-IN" else len(party_idx)
    rec = {"run_id": run_id_for(ds, method, mode, partition_mode, alpha, n_clients, seed),
           "method": method, "dataset": ds["name"], "mode": mode,
           "partition": "pooled" if method == "ALL-IN" else partition_mode,
           "alpha": None if partition_mode == "iid" or method == "ALL-IN" else alpha,
           "n_parties": n_clients, "seed": seed,
           "config": {k: cfg[k] for k in DEFAULTS}, "environment": ENV,
           "data": {"label_col": ds["label_col"], "n_train": len(y_tr), "n_test": len(ds["y_test"]),
                    "n_features": ds["n_features"], "drop_cols": ds["drop_cols"],
                    "class_names": class_names_for_mode(ds, mode),
                    "party_sizes": [len(i) for i in party_idx] if method != "ALL-IN" else [len(y_tr)],
                    "party_class_counts": [np.bincount(y_tr[i], minlength=K).tolist() for i in party_idx]
                    if method != "ALL-IN" else []}}
    comm = ZERO_COMM
    if method == "ALL-IN":
        model, timing = train_xgb_local(ds["X_train"], y_tr, mode, K, cfg, seed)
        q, inf, ms, y_pred = evaluate(model, mode, ds, cfg)
    elif method == "SimFL":
        model, timing, comm, extra = train_simfl([(ds["X_train"][i], y_tr[i]) for i in party_idx], mode, K, cfg,
                                                 seed, log_every)
        rec["simfl"] = extra
        q, inf, ms, y_pred = evaluate(model, mode, ds, cfg)
    elif method == "SOLO":
        per = []
        for i in party_idx:
            m, tm = train_xgb_local(ds["X_train"][i], y_tr[i], mode, K, cfg, seed)
            per.append((tm, *evaluate(m, mode, ds, cfg)))
        timing = mean_records([p[0] for p in per])
        q, inf, ms = (mean_records([p[j] for p in per]) for j in (1, 2, 3))
        rec["solo_per_party"] = [{"macro_f1": p[1]["macro_f1"], "accuracy": p[1]["accuracy"], "fpr": p[1]["fpr"],
                                  "train_total_s": p[0]["train_total"]} for p in per]
        model, y_pred = None, None
    else:
        raise ValueError(method)
    rec["timing"] = {f"{k}_s": float(timing.get(k, 0.0)) for k in TIMING_KEYS}
    rec["communication"] = comm
    rec["network"] = network_estimate(comm, cfg)
    rec["quality"], rec["inference"], rec["model"] = q, inf, ms
    return rec, model, y_pred


def results_dir(ds_name):
    d = RESULTS_DIR / ds_name / "runs"
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_record(rec):
    (results_dir(rec["dataset"]) / f"{rec['run_id']}.json").write_text(json.dumps(to_jsonable(rec), indent=1))


def flatten(d, prefix=""):
    row = {}
    for k, v in d.items():
        if k in ("confusion_matrix", "party_class_counts", "solo_per_party", "environment", "support"):
            continue
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            row.update(flatten(v, key + "."))
        elif isinstance(v, list):
            row[key] = json.dumps(v)
        else:
            row[key] = v
    return row


def rebuild_csv(ds_name=None):
    """Rebuild runs.csv for one dataset (or all datasets when ds_name is None)."""
    names = [ds_name] if ds_name else [p.name for p in RESULTS_DIR.iterdir() if (p / "runs").is_dir()]
    frames = []
    for n in names:
        recs = [json.loads(p.read_text()) for p in sorted((RESULTS_DIR / n / "runs").glob("*.json"))]
        df = pd.DataFrame([flatten(r) for r in recs])
        df.to_csv(RESULTS_DIR / n / "runs.csv", index=False)
        frames.append(df)
    all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if ds_name is None and len(all_df):
        all_df.to_csv(RESULTS_DIR / "all_runs.csv", index=False)
    return all_df

# Full pipeline

`run_simfl(df, ...)` mirrors `run_fedtree`: it prepares the data, partitions the training set, trains SimFL, prints the
results, and optionally trains the SOLO / ALL-IN baselines on the same partition for comparison. All runs are saved.

By default it uses the first grid seed (`GRID["seeds"][0]`, i.e. seed 0) and the same split, partition and settings as the
grid, so each per-dataset run **is** the corresponding grid run (same `run_id`). The full grid then skips it, and a
baseline already saved (e.g. ALL-IN from the IID cell) is reused rather than retrained.

In [ ]:
def run_simfl(
    df: pd.DataFrame,
    dataset_name: str,
    label_col: str = "Label",
    benign_label=None,                    # e.g. 0 / "Normal"; enables binary mode and FPR
    class_names: Optional[dict] = None,
    drop_cols=(),
    mode: str = "multiclass",             # "multiclass" | "binary"
    partition_mode: str = "iid",          # "iid" | "dirichlet"
    n_clients: int = 10,
    dirichlet_alpha: float = 0.5,
    seed: Optional[int] = None,           # None -> GRID["seeds"][0], so this run is also a grid run
    compare_baselines: bool = True,
    verbose: bool = True,
    **overrides,                          # any key of DEFAULTS, e.g. n_trees=50, max_depth=8, lsh_r=4.0
) -> Dict:
    cfg = {**DEFAULTS, **overrides}
    seed = GRID["seeds"][0] if seed is None else seed
    ds = prepare_dataset(df, dataset_name, label_col, benign_label, drop_cols, class_names, cfg)
    party_idx = make_partition(ds["y_train"], partition_mode, n_clients, seed, dirichlet_alpha, cfg)
    names = class_names_for_mode(ds, mode)

    if verbose:
        part = partition_mode if partition_mode == "iid" else f"dirichlet(alpha={dirichlet_alpha})"
        print(f"Dataset: {dataset_name} {df.shape} | mode={mode} | partition={part} | clients={n_clients}")
        y_tr = labels_for_mode(ds["y_train"], mode)
        for c, i in enumerate(party_idx):
            cnt = Counter(y_tr[i].tolist())
            print(f"  Client {c}: {len(i)} samples, classes={ {names[k]: cnt[k] for k in sorted(cnt)} }")

    rec, model, y_pred = run_method("SimFL", ds, mode, partition_mode, dirichlet_alpha, party_idx, seed, cfg,
                                    log_every=10 if verbose else 0)
    save_record(rec)
    records = {"SimFL": rec}
    if compare_baselines:
        for m in ["SOLO", "ALL-IN"]:
            path = results_dir(dataset_name) / (run_id_for(ds, m, mode, partition_mode, dirichlet_alpha,
                                                           1 if m == "ALL-IN" else n_clients, seed) + ".json")
            if GRID["resume"] and path.exists():          # e.g. ALL-IN already trained by the previous cell
                records[m] = json.loads(path.read_text())
            else:
                records[m], _, _ = run_method(m, ds, mode, partition_mode, dirichlet_alpha, party_idx, seed, cfg)
                save_record(records[m])
    rebuild_csv(dataset_name)

    if verbose:
        q, tm, cm, nw, inf, ms = (rec[k] for k in ["quality", "timing", "communication", "network", "inference", "model"])
        print("\n===== RESULTS (SimFL) =====")
        print(f"Accuracy     : {100 * q['accuracy']:.2f}%")
        print(f"F1-macro     : {100 * q['macro_f1']:.2f}%")
        print(f"F1-weighted  : {100 * q['weighted_f1']:.2f}%")
        if q["fpr"] is not None:
            print(f"FPR (benign) : {100 * q['fpr']:.2f}%")
        print("\n===== EFFICIENCY =====")
        print(f"Train time (total)        : {tm['train_total_s']:.1f} s")
        print(f"  LSH preprocessing       : {tm['lsh_preprocessing_s']:.2f} s")
        print(f"  local gradients         : {tm['local_gradients_s']:.2f} s")
        print(f"  tree building           : {tm['tree_building_s']:.2f} s")
        print(f"  aggregation / comm      : {tm['aggregation_comm_s']:.2f} s")
        print(f"Est. network time         : {nw['total_s']:.2f} s ({nw['latency_ms']} ms latency, {nw['bandwidth_mbps']} Mbps)")
        print(f"Test time / sample (batch): {1e3 * inf['batch_per_sample_s']:.4f} ms")
        print(f"Single-sample latency     : {1e3 * inf['single_median_s']:.4f} ms (median of {inf['n_single']})")
        print(f"Model size                : {ms['bytes'] / 2**20:.3f} MB | {ms['n_trees']} trees | {ms['total_nodes']} nodes")
        print(f"Comm rounds               : {cm['preprocessing_rounds']} preprocessing ({cm['hash_exchange_rounds']} hash exchange)"
              f" + {cm['training_rounds']} training")
        print(f"Communication             : {cm['preprocessing_bytes'] / 2**20:.1f} MB preprocessing"
              f" + {cm['training_bytes'] / 2**20:.1f} MB training = {cm['total_bytes'] / 2**20:.1f} MB")
        print(f"LSH: L={rec['simfl']['L']}, r={rec['simfl']['r']}, mean matching hashes="
              f"{rec['simfl']['mean_matching_hashes']:.1f}, unmatched samples={rec['simfl']['unmatched_samples']}")
        print("\nClassification Report:")
        print(classification_report(labels_for_mode(ds["y_test"], mode), y_pred, labels=list(range(len(names))),
                                    target_names=names, digits=4, zero_division=0))
        if compare_baselines:
            rows = [{"method": m, "accuracy": r["quality"]["accuracy"], "macro_f1": r["quality"]["macro_f1"],
                     "weighted_f1": r["quality"]["weighted_f1"], "fpr": r["quality"]["fpr"],
                     "train_s": r["timing"]["train_total_s"], "comm_MB": r["communication"]["total_bytes"] / 2**20}
                    for m, r in records.items()]
            print("===== SimFL vs baselines (SOLO = mean over clients) =====")
            display(pd.DataFrame(rows).set_index("method").round(4))

    return {"records": records, "model": model, "dataset": ds, "y_test": labels_for_mode(ds["y_test"], mode),
            "y_pred": y_pred}

# Experiment grid

`run_experiments` runs every `mode × partition × n_clients × seed` combination in `GRID` for SimFL and SOLO, plus ALL-IN
once per `mode × seed`. With the default `GRID` this is multi-class, IID + Dirichlet α=0.5, 10 clients, seeds 0–2
(seed-0 runs already come from the per-dataset cells). Runs whose JSON already exists are skipped (`GRID["resume"]`), so an interrupted Colab session
can simply be re-run.

In [ ]:
def run_experiments(df, dataset_name, label_col="Label", benign_label=None, class_names=None, drop_cols=(),
                    grid=None, **overrides):
    grid = {**GRID, **(grid or {})}
    cfg = {**DEFAULTS, **overrides}
    ds = prepare_dataset(df, dataset_name, label_col, benign_label, drop_cols, class_names, cfg)
    modes = [m for m in grid["modes"] if m == "multiclass" or ds["has_benign"]]
    if len(modes) < len(grid["modes"]):
        print(f"[{dataset_name}] no benign_label -> binary mode skipped")
    plan = ([("iid", None)] if "iid" in grid["partitions"] else []) + \
           ([("dirichlet", a) for a in grid["dirichlet_alphas"]] if "dirichlet" in grid["partitions"] else [])

    t0, done, skipped = time.perf_counter(), 0, 0
    for seed in grid["seeds"]:
        jobs = [("ALL-IN", mode, "iid", None, [np.arange(len(ds["y_train"]))])
                for mode in modes if "ALL-IN" in grid["methods"]]
        for part, alpha in plan:
            for M in grid["n_parties"]:
                idx = make_partition(ds["y_train"], part, M, seed, alpha, cfg)
                jobs += [(m, mode, part, alpha, idx) for mode in modes for m in grid["methods"] if m != "ALL-IN"]
        for method, mode, part, alpha, idx in jobs:
            rid = run_id_for(ds, method, mode, part, alpha, 1 if method == "ALL-IN" else len(idx), seed)
            if grid["resume"] and (results_dir(dataset_name) / f"{rid}.json").exists():
                skipped += 1
                continue
            rec, _, _ = run_method(method, ds, mode, part, alpha, idx, seed, cfg)
            save_record(rec)
            done += 1
            print(f"[{time.perf_counter() - t0:7.0f}s] {rid:<60} macroF1={rec['quality']['macro_f1']:.4f} "
                  f"train={rec['timing']['train_total_s']:.1f}s comm={rec['communication']['total_bytes'] / 2**20:.1f}MB")
    print(f"[{dataset_name}] {done} new runs, {skipped} skipped")
    return rebuild_csv(dataset_name)

# Sanity checks (≈30 s)

These run on synthetic data, so no dataset is needed:

1. The LSH search finds the true maximum and breaks ties uniformly.
2. The communicator delivers copies and counts bytes exactly.
3. **SimFL with a single party equals centralized XGBoost**, in both modes.
4. The merged global model equals the sum of its one-round trees.

In [ ]:
_rng = np.random.default_rng(1)
_Hq, _Ht = _rng.integers(0, 3, (400, 6)), _rng.integers(0, 3, (300, 6))
_idx, _best = most_similar(_Hq, _Ht, _rng, chunk=37)
_C = (_Hq[:, None, :] == _Ht[None, :, :]).sum(2)
assert (_C[np.arange(400), _idx] == _C.max(1)).all() and (_best == _C.max(1)).all()
_ties = np.flatnonzero(_C[0] == _C[0].max())
_picks = np.array([most_similar(_Hq[:1], _Ht, np.random.default_rng(s))[0][0] for s in range(1500)])
print("1. LSH argmax ok; tie counts", [int((_picks == t).sum()) for t in _ties], "expected ~", 1500 // len(_ties))

_c = Communicator(2); _c.new_round("training", "x"); _a = np.ones(10, np.float32)
_c.send(0, 1, "t", {"a": _a}); _a[:] = 5
(_, _got), = _c.receive(1, "t")
assert _got["a"].sum() == 10 and _c.summary()["training_bytes"] == 40
print("2. communicator copies payloads and counts bytes exactly")

_X = _rng.normal(size=(3000, 12)).astype(np.float32)
_y = ((_X[:, 0] + _X[:, 1] > 0).astype(int) + (_X[:, 2] > 1)).astype(np.int64)
_cfg = {**DEFAULTS, "n_trees": 8}
for _mode, _yy, _K in [("binary", (_y > 0).astype(np.int64), 2), ("multiclass", _y, 3)]:
    _m1, *_ = train_simfl([(_X, _yy)], _mode, _K, _cfg, seed=0)
    _m2, _ = train_xgb_local(_X, _yy, _mode, _K, _cfg, seed=0)
    _d = np.abs(_m1.inplace_predict(_X, predict_type="margin") - _m2.inplace_predict(_X, predict_type="margin")).max()
    print(f"3. [{_mode}] max |SimFL(1 party) - XGBoost| margin diff = {_d:.2e}")
    assert _d < 1e-4
print(f"4. global model type: {type(_m1).__name__} (merged single booster)" if isinstance(_m1, xgb.Booster)
      else "4. using SummedEnsemble fallback")

# Experiments

# 1. CICIDS2017

In [ ]:
CICIDS = dict(
    dataset_name="CICIDS2017", label_col="Label", benign_label=0,
    class_names={0: "BENIGN", 1: "Bot", 2: "BruteForce", 3: "DoS", 4: "Infiltration", 5: "PortScan", 6: "WebAttack"},
)
df = pd.read_csv("CICIDS2017_sample_km.csv")
print(df.shape)

# IID

In [ ]:
# ----- IID ----- (seed 0: shared with the grid)
res_cic_iid = run_simfl(df, **CICIDS, mode="multiclass", partition_mode="iid", n_clients=10)

# Non-IID (Dirichlet α = 0.5)

In [ ]:
# ----- Non-IID (Dirichlet) ----- (seed 0: shared with the grid)
res_cic_noniid = run_simfl(df, **CICIDS, mode="multiclass", partition_mode="dirichlet", n_clients=10,
                           dirichlet_alpha=0.5)

# Full grid

In [ ]:
if RUN_FULL_GRID:
    grid_cic = run_experiments(df, **CICIDS)

# 2. IoTID2020

In [ ]:
df = pd.read_csv("IoT_Network_Intrusion_Dataset_0.1.csv")

In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

le = LabelEncoder()
df["Cat"] = le.fit_transform(df["Cat"])
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

# 2. Columns that should NOT be normalized
exclude_from_norm = ["Cat"]

# 3. All other columns -> normalize
feature_cols = [c for c in df.columns if c not in exclude_from_norm]
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

IOT = dict(
    dataset_name="IoTID2020", label_col="Cat",
    benign_label=int(le.transform(["Normal"])[0]),                     # benign class for binary mode / FPR
    class_names=dict(zip(range(len(le.classes_)), le.classes_)),
)

# IID

In [ ]:
# ----- IID ----- (seed 0: shared with the grid)
res_iot_iid = run_simfl(df, **IOT, mode="multiclass", partition_mode="iid", n_clients=10)

# Non-IID (Dirichlet α = 0.5)

In [ ]:
# ----- Non-IID (Dirichlet) ----- (seed 0: shared with the grid)
res_iot_noniid = run_simfl(df, **IOT, mode="multiclass", partition_mode="dirichlet", n_clients=10,
                           dirichlet_alpha=0.5)

# Full grid

In [ ]:
if RUN_FULL_GRID:
    grid_iot = run_experiments(df, **IOT)

# 3. CICEVSE2024

Labels are already integer-encoded (12 classes). **Set `CICEVSE_BENIGN` to the id of the benign class** to get the FPR
(and binary runs, if `"binary"` is added to `GRID["modes"]`). While it is `None`, the FPR is reported as `None`.

In [ ]:
CICEVSE_BENIGN = None          # <-- set to the benign class id, e.g. CICEVSE_BENIGN = 0
CICEVSE = dict(dataset_name="CICEVSE2024", label_col="Label", benign_label=CICEVSE_BENIGN)
df = pd.read_csv("CICEVSE2024_12classes_sample.csv")
print(df.shape, "|", df["Label"].value_counts().sort_index().to_dict())

# IID

In [ ]:
# ----- IID ----- (seed 0: shared with the grid)
res_evse_iid = run_simfl(df, **CICEVSE, mode="multiclass", partition_mode="iid", n_clients=10)

# Non-IID (Dirichlet α = 0.5)

In [ ]:
# ----- Non-IID (Dirichlet) ----- (seed 0: shared with the grid)
res_evse_noniid = run_simfl(df, **CICEVSE, mode="multiclass", partition_mode="dirichlet", n_clients=10,
                            dirichlet_alpha=0.5)

# Full grid

In [ ]:
if RUN_FULL_GRID:
    grid_evse = run_experiments(df, **CICEVSE)

# Final comparison

All saved runs, from every dataset, are summarized as mean ± std over seeds.

In [ ]:
runs = rebuild_csv()
runs["setting"] = np.where(runs["partition"] == "dirichlet", "dirichlet α=" + runs["alpha"].astype(str), runs["partition"])
metrics = {"quality.macro_f1": "macro-F1", "quality.weighted_f1": "weighted-F1", "quality.accuracy": "accuracy",
           "quality.fpr": "FPR", "timing.train_total_s": "train s", "network.total_s": "est. net s",
           "communication.total_bytes": "bytes", "communication.total_rounds": "rounds",
           "model.bytes": "model bytes", "inference.batch_per_sample_s": "batch s/sample",
           "inference.single_median_s": "single s"}
summary = runs.groupby(["dataset", "mode", "method", "setting", "n_parties"])[list(metrics)].agg(["mean", "std"])
summary.columns = [f"{metrics[a]} {b}" for a, b in summary.columns]
summary.to_csv(RESULTS_DIR / "summary.csv")
with pd.option_context("display.max_rows", 500, "display.max_columns", 30, "display.width", 250):
    display(summary[[c for c in summary.columns if c.endswith("mean")] + ["macro-F1 std"]].round(4))

In [ ]:
# Share of the SOLO -> ALL-IN gap that SimFL closes: (SimFL - SOLO) / (ALL-IN - SOLO)
key = ["dataset", "mode", "setting", "n_parties"]
fed = runs[runs["method"] != "ALL-IN"]
f1 = fed.groupby(key + ["method"])["quality.macro_f1"].mean().unstack("method")
allin = runs[runs["method"] == "ALL-IN"].groupby(["dataset", "mode"])["quality.macro_f1"].mean()
f1["ALL-IN"] = [allin.get((d, m), np.nan) for d, m, *_ in f1.index]
if {"SimFL", "SOLO"} <= set(f1.columns):
    f1["gap_closed"] = (f1["SimFL"] - f1["SOLO"]) / (f1["ALL-IN"] - f1["SOLO"])
display(f1.round(4))

In [ ]:
import matplotlib.pyplot as plt

STYLE = {"SimFL": dict(color="#2a78d6", marker="o", ls="-"),      # blue
         "SOLO": dict(color="#eb6834", marker="s", ls="--"),      # orange
         "ALL-IN": dict(color="#1baf7a", marker=None, ls=":")}    # aqua


def plot_metric(col, ylabel, methods=("SimFL", "SOLO", "ALL-IN"), logy=False, fname=None):
    fed = runs[runs["method"] != "ALL-IN"]
    settings = [s for s in ["iid", "dirichlet α=1.0", "dirichlet α=0.5", "dirichlet α=0.1"] if s in set(fed["setting"])]
    panels = [(d, m) for d in runs["dataset"].unique() for m in ["binary", "multiclass"]
              if ((runs["dataset"] == d) & (runs["mode"] == m)).any()]
    fig, axes = plt.subplots(len(panels), len(settings), figsize=(3.4 * len(settings), 2.7 * len(panels)),
                             sharex=True, squeeze=False)
    for r, (d, m) in enumerate(panels):
        sub = runs[(runs["dataset"] == d) & (runs["mode"] == m)]
        for c, s in enumerate(settings):
            ax = axes[r, c]
            for meth in methods:
                if meth == "ALL-IN":
                    v = sub[sub["method"] == "ALL-IN"][col]
                    if len(v):
                        ax.axhline(v.mean(), lw=2, label="ALL-IN", color=STYLE[meth]["color"], ls=STYLE[meth]["ls"])
                    continue
                g = sub[(sub["method"] == meth) & (sub["setting"] == s)].groupby("n_parties")[col].agg(["mean", "std"])
                if len(g):
                    ax.errorbar(g.index, g["mean"], yerr=g["std"].fillna(0), lw=2, ms=7, capsize=3, label=meth, **STYLE[meth])
            ax.set_xticks(sorted(fed["n_parties"].unique()))
            if logy: ax.set_yscale("log")
            ax.grid(alpha=0.25, lw=0.6)
            for side in ("top", "right"): ax.spines[side].set_visible(False)
            if r == 0: ax.set_title(s, fontsize=10)
            if c == 0: ax.set_ylabel(f"{d} · {m}\n{ylabel}", fontsize=9)
            if r == len(panels) - 1: ax.set_xlabel("number of clients")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.tight_layout()
    if fname: fig.savefig(RESULTS_DIR / fname, dpi=130, bbox_inches="tight")
    plt.show()


plot_metric("quality.macro_f1", "macro-F1", fname="macro_f1.png")
plot_metric("communication.total_bytes", "bytes sent", methods=("SimFL",), logy=True, fname="bytes.png")
plot_metric("timing.train_total_s", "training time (s)", fname="train_time.png")

# Download results

In [ ]:
import shutil
shutil.make_archive("simfl_results", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("simfl_results.zip")
except ImportError:
    print("saved simfl_results.zip")